# Hafta 11 — Boyut İndirgeme ve Öznitelik Seçimi

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy umap-learn

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

Bir müşteri hakkında yüz farklı bilgi tutulduğunu düşünelim. Bu bilgilerin önemli bir kısmı birbirine çok benzer; örneğin evin metrekaresi ile oda sayısı neredeyse aynı şeyi anlatır. Boyut azaltma yöntemleri bu kadar çok bilgiyi, az kayıpla iki ya da üç özet bilgiye indirir.

Üç boyutlu bir nesnenin duvara düşen gölgesi bu işe iyi bir örnektir. Işığı doğru açıdan tutarsanız gölge, nesnenin şeklini oldukça iyi yansıtır. Aynı şekilde yüzlerce boyutlu veriler de ekranda iki boyutlu bir grafikle incelenebilir.

---

## Bu Haftanın Hedefleri

- PCA'nın çalışma mantığını özdeğer ve özvektörlerle anlamak
- Açıklanan varyans oranıyla bileşen sayısını seçmek
- t-SNE ve UMAP grafiklerini doğru okumak
- Öznitelik seçimi yöntemlerini tanımak

---

## 1. Temel Bileşen Analizi (PCA)

### Fikir

Öğrencilerin matematik ve fizik notları birbiriyle güçlü biçimde ilişkilidir; noktalar neredeyse tek bir doğru boyunca uzanır. Her öğrenciyi iki not yerine bu doğru üzerindeki **tek bir konumla** göstersek çok az bilgi kaybederiz.

```
 Fizik
   │            ● ●        PC1: verinin en çok yayıldığı yön
   │         ● ●  ╱
   │      ● ● ╱ ●
   │    ●  ╱ ●             PC2: PC1'e dik, az yayılmış yön
   │     ╱                     ↖
   └────────────────── Matematik
```

PCA, verinin **en çok yayıldığı (varyansı en büyük) yönleri** bulur. Bu yönlere **temel bileşen** denir. Varyans önemlidir, çünkü herkesin aynı değere sahip olduğu bir yön, öğrencileri birbirinden ayırt etmeye yardım etmez.

### Algoritma

```
1. Veriyi merkezle (her sütundan ortalamasını çıkar)
2. Kovaryans matrisini hesapla
3. Özvektörleri (yönler) ve özdeğerleri (o yöndeki varyans) bul
4. Özdeğeri en büyük k özvektörü seç
5. Veriyi bu yönlere izdüşür
```

### Elle örnek

10 noktalık 2 boyutlu bir veride kovaryans matrisi şöyle çıkıyor:

```
     ┌ 0.617   0.615 ┐
Σ =  └ 0.615   0.717 ┘
```

Bu matrisin özdeğerleri **λ₁ = 1.284** ve **λ₂ = 0.049**'dur. Birinci bileşenin yönü (özvektörü) **(0.68, 0.74)**'tür.

```
Açıklanan varyans:  PC1 = 1.284 / (1.284 + 0.049) = %96.3     PC2 = %3.7
```

Verinin %96'sı tek bir boyutla açıklanıyor. İki boyutu bire indirirsek bilginin yalnızca %4'ünü kaybederiz.

### Sıfırdan PCA

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def pca(X, k):
    Xc = X - X.mean(axis=0)                                 # 1) merkezle
    kov = Xc.T @ Xc / (len(X) - 1)                          # 2) kovaryans
    ozdeger, ozvektor = np.linalg.eigh(kov)                 # 3) özdeğer ve özvektörler
    sira = np.argsort(ozdeger)[::-1]                        # 4) büyükten küçüğe sırala
    W = ozvektor[:, sira[:k]]
    oran = ozdeger[sira[:k]] / ozdeger.sum()
    return Xc @ W, oran                                     # 5) izdüşür

X = np.array([[2.5, 2.4], [0.5, 0.7], [2.2, 2.9], [1.9, 2.2], [3.1, 3.0],
              [2.3, 2.7], [2.0, 1.6], [1.0, 1.1], [1.5, 1.6], [1.1, 0.9]])
Z, oran = pca(X, k=1)
print("Açıklanan varyans:", oran.round(3))                  # [0.963]

from sklearn.decomposition import PCA
print(PCA(n_components=1).fit(X).explained_variance_ratio_)  # aynı sonuç

### Bileşen sayısını seçmek

Genellikle varyansın %90–95'ini koruyan en küçük bileşen sayısı seçilir:

In [ ]:
from sklearn.datasets import load_digits

Xd, yd = load_digits(return_X_y=True)                       # 8×8 = 64 piksellik rakam görüntüleri
kumulatif = np.cumsum(PCA().fit(Xd).explained_variance_ratio_)
plt.plot(range(1, 65), kumulatif, "o-", ms=3); plt.axhline(0.95, color="r", ls="--")
plt.xlabel("Bileşen sayısı"); plt.ylabel("Kümülatif açıklanan varyans"); plt.show()
print("%95 için gereken bileşen:", np.argmax(kumulatif >= 0.95) + 1)

> PCA'dan önce, birimleri farklı öznitelikler **standardize edilmelidir**. PCA doğrusal bir yöntemdir ve sınıf etiketlerini kullanmaz.

---

## 2. t-SNE ve UMAP ile Görselleştirme

PCA verinin genel yapısını korur. **t-SNE** ve **UMAP** ise **komşulukları** korumaya odaklanır: Yüksek boyutta birbirine yakın olan noktalar, iki boyutlu grafikte de yakın durur. Bu yüzden kümeleri görselleştirmede çok başarılıdırlar.

**t-SNE'nin çalışma mantığı:** Yüksek boyutta her nokta çifti için bir "komşu olma olasılığı" hesaplanır. Noktalar iki boyutta rastgele yerleştirilir ve bu olasılıklar mümkün olduğunca korunacak şekilde adım adım kaydırılır. Komşular birbirini çeker, komşu olmayanlar birbirini iter.

**Grafiği okurken dikkat edilmesi gerekenler:**

| Grafikte gördüğünüz | Anlamlı mı? |
|---|---|
| Hangi noktaların aynı kümede olduğu | Evet |
| Kümelerin büyüklüğü | Hayır |
| Kümeler arası mesafeler | Güvenilmez |
| Eksen değerleri | Anlamsız |

t-SNE yalnızca görselleştirme içindir; yeni bir noktayı mevcut grafiğe yerleştiremez. UMAP ise daha hızlıdır ve yeni verilere de uygulanabilir.

In [ ]:
from sklearn.manifold import TSNE

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
for a, (ad, Z) in zip(ax, [("PCA", PCA(2).fit_transform(Xd)),
                           ("t-SNE", TSNE(2, random_state=0).fit_transform(Xd))]):
    a.scatter(Z[:, 0], Z[:, 1], c=yd, cmap="tab10", s=5); a.set_title(ad)
plt.show()

---

## 3. Öznitelik Seçimi

Boyut indirgemenin aksine öznitelik seçimi **orijinal sütunları korur**; yalnızca gereksiz olanları atar. Böylece model daha anlaşılır olur.

| Yöntem | Nasıl çalışır? | Örnek |
|---|---|---|
| **Filtre** | Her sütunu tek tek bir istatistikle puanlar | Varyansı sıfır olanları at, hedefle ilişkiyi ölç |
| **Sarmalayıcı** | Sütun gruplarını modeli eğiterek dener | İleri seçim: her adımda en çok faydayı sağlayan sütunu ekle |
| **Gömülü** | Seçim, modelin eğitimi sırasında yapılır | Lasso, ağaç tabanlı öznitelik önemleri |

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

veri = load_breast_cancer()
Xb = StandardScaler().fit_transform(veri.data)

filtre = SelectKBest(mutual_info_classif, k=5).fit(Xb, veri.target)
print("Filtre    :", veri.feature_names[filtre.get_support()])

sarmalayici = RFE(LogisticRegression(max_iter=5000), n_features_to_select=5).fit(Xb, veri.target)
print("Sarmalayıcı:", veri.feature_names[sarmalayici.support_])

> Öznitelik seçimi de veriden öğrenir. Sızıntıyı önlemek için Pipeline içinde yapılmalıdır.

---

## 4. Laboratuvar: MNIST Üzerinde PCA ve t-SNE

In [ ]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False, parser="auto")
idx = np.random.default_rng(0).choice(len(mnist.data), 5000, replace=False)
X, y = mnist.data[idx] / 255.0, mnist.target[idx].astype(int)    # 5000 görüntü, 784 piksel

Z_pca = PCA(2).fit_transform(X)
Z_tsne = TSNE(2, random_state=0).fit_transform(PCA(50).fit_transform(X))   # önce 50 boyuta indir
fig, ax = plt.subplots(1, 2, figsize=(16, 7))
for a, Z, ad in [(ax[0], Z_pca, "PCA"), (ax[1], Z_tsne, "t-SNE")]:
    a.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=3); a.set_title(ad)
plt.show()

**Görevler:**
1. MNIST'te varyansın %95'i için kaç bileşen gerekiyor? 784 boyuttan ne kadarlık bir sıkıştırma bu?
2. Bir rakamı 10, 50 ve 150 bileşenle yeniden oluşturup (`inverse_transform`) orijinaliyle karşılaştırın.
3. t-SNE grafiğinde hangi rakamlar birbirine karışıyor? Bu rakamlar görsel olarak da benzer mi?

---

## 5. Özet

- **PCA**, verinin en çok yayıldığı yönleri bulur; özdeğerler her yöndeki varyansı gösterir.
- Bileşen sayısı, açıklanan varyansın %90–95'ine göre seçilir.
- **t-SNE ve UMAP** komşulukları koruyan görselleştirme yöntemleridir; küme büyüklükleri ve aralarındaki mesafeler yorumlanmamalıdır.
- Öznitelik seçimi orijinal sütunları korur: filtre, sarmalayıcı ve gömülü yöntemler.

## Kendinizi Sınayın

1. Özdeğerleri 4 ve 1 olan bir veride birinci bileşen varyansın yüzde kaçını açıklar?
2. PCA'dan önce neden standardizasyon yapılır?
3. t-SNE neden bir sınıflandırma modelinden önce ön işleme adımı olarak kullanılmaz?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.